# T18 — ReAct Agent for Multi-Step Reasoning

## Objective
Build a ReAct-pattern agent (Reasoning + Acting) to solve multi-step reasoning tasks. Document and log the step-by-step reasoning traces.

### ReAct Framework
```
User Question
    │
    ▼
┌─────────┐
│ Thought │ ──> Reason about what step to take next
└────┬────┘
     │
     ▼
┌─────────┐
│ Action  │ ──> Select tool and prepare arguments
└────┬────┘
     │
     ▼
┌──────────────┐
│ Action Input │ ──> Pass inputs to the selected tool
└────┬─────────┘
     │
     ▼
┌─────────────┐
│ Observation │ ──> Receive tool execution result
└────┬────────┘
     │
     ▼
( Repeat until problem is solved )
     │
     ▼
┌──────────────┐
│ Final Answer │ ──> Provide complete answer to user
└──────────────┘
```

### Key Components
1. **Custom Tools**: Knowledge Base Search, Calculator, and Unit/Currency Converter.
2. **ReAct Prompt Engine**: Prompt template instructing the model to think before acting.
3. **Execution Loop**: Parser to extract Action/Action Input, execute tool, append Observation, and repeat.
4. **Reasoning Trace Logger**: Formatted visual output for every step of the reasoning chain.
5. **Multi-Step Test Cases**: Evaluating 3 complex, multi-hop user queries.


## 1. Environment Setup & Imports


In [1]:
import os
import re
import json
import ast
import operator
from dotenv import load_dotenv
from openai import OpenAI

# Load API key from .env file located in parent directory
load_dotenv(dotenv_path=os.path.join("..", ".env"), override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    load_dotenv(override=True)
    api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment or .env file.")

client = OpenAI(api_key=api_key)
print("OpenAI client successfully initialized!")


OpenAI client successfully initialized!


## 2. Define Custom Tools for Multi-Step Reasoning


In [2]:
# -------------------------------------------------------------
# Tool 1: Knowledge Base Search Tool
# -------------------------------------------------------------
KNOWLEDGE_BASE = {
    "apple": "Apple Inc. reported a Q4 revenue of $89.5 billion and net income of $23.0 billion.",
    "microsoft": "Microsoft Corporation reported a Q4 revenue of $56.5 billion and net income of $22.3 billion.",
    "tesla": "Tesla Inc. reported a Q4 revenue of $25.17 billion and net income of $7.9 billion.",
    "tokyo": "Tokyo is the capital of Japan. The population of Tokyo is approximately 14 million people.",
    "paris": "Paris is the capital of France. The distance between Paris and London is 344 kilometers.",
    "mumbai": "Mumbai is the financial capital of India. The distance between Mumbai and Pune is 150 kilometers."
}

def search_knowledge_base(query: str) -> str:
    """Searches local knowledge base for facts and data."""
    query_lower = query.lower()
    results = []
    for key, value in KNOWLEDGE_BASE.items():
        if key in query_lower:
            results.append(value)
    if results:
        return " | ".join(results)
    return f"No direct entry found for query '{query}'. Available topics: {list(KNOWLEDGE_BASE.keys())}"

# -------------------------------------------------------------
# Tool 2: Safe Mathematical Calculator
# -------------------------------------------------------------
allowed_operators = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
    ast.USub: operator.neg
}

def _evaluate_ast(node):
    if isinstance(node, ast.Constant):
        return node.value
    elif isinstance(node, ast.BinOp):
        left = _evaluate_ast(node.left)
        right = _evaluate_ast(node.right)
        op = allowed_operators.get(type(node.op))
        if op is None:
            raise ValueError("Unsupported operation")
        return op(left, right)
    elif isinstance(node, ast.UnaryOp):
        operand = _evaluate_ast(node.operand)
        op = allowed_operators.get(type(node.op))
        if op is None:
            raise ValueError("Unsupported operation")
        return op(operand)
    else:
        raise ValueError("Invalid mathematical expression")

def calculator(expression: str) -> str:
    """Safely evaluates math expressions (e.g. '89.5 - 56.5' or '344 * 0.621371')."""
    try:
        clean_expr = expression.replace(" ", "").replace("^", "**")
        parsed = ast.parse(clean_expr, mode='eval')
        result = _evaluate_ast(parsed.body)
        return str(result)
    except Exception as e:
        return f"Error evaluating expression '{expression}': {str(e)}"

# -------------------------------------------------------------
# Tool 3: Unit & Currency Converter Tool
# -------------------------------------------------------------
def unit_converter(val_unit_to_unit: str) -> str:
    """Converts units. Example input format: '150 km to miles' or '81.67 USD to INR'"""
    try:
        parts = val_unit_to_unit.lower().split()
        if "km" in parts and "miles" in parts:
            val = float(parts[0])
            miles = val * 0.621371
            return f"{val} km = {miles:.2f} miles"
        elif "miles" in parts and "km" in parts:
            val = float(parts[0])
            km = val / 0.621371
            return f"{val} miles = {km:.2f} km"
        elif "usd" in parts and "inr" in parts:
            val = float(parts[0])
            inr = val * 83.5
            return f"{val} USD = {inr:.2f} INR"
        else:
            return f"Conversion for '{val_unit_to_unit}' unsupported. Supported: 'km to miles', 'miles to km', 'usd to inr'."
    except Exception as e:
        return f"Conversion error: {str(e)}"

AVAILABLE_TOOLS = {
    "search_knowledge_base": search_knowledge_base,
    "calculator": calculator,
    "unit_converter": unit_converter
}

print("Custom tools initialized:")
for name in AVAILABLE_TOOLS:
    print(f" - {name}")


Custom tools initialized:
 - search_knowledge_base
 - calculator
 - unit_converter


## 3. ReAct System Prompt & Reasoning Engine


In [3]:
REACT_SYSTEM_PROMPT = """You are an AI assistant designed to solve multi-step reasoning problems using the ReAct (Reasoning + Acting) framework.

You have access to the following tools:

1. search_knowledge_base(query: str): Searches the local knowledge base for facts about companies, cities, revenue, distance, etc.
2. calculator(expression: str): Evaluates math expressions like '89.5 - 56.5' or '344 * 0.621371'.
3. unit_converter(val_unit_to_unit: str): Converts units such as '150 km to miles' or '81.67 USD to INR'.

### Format Instructions

To answer questions, you MUST follow this strict step-by-step format:

Question: the input question you must answer
Thought: reason step-by-step about what you need to do next
Action: the tool name to use (must be one of [search_knowledge_base, calculator, unit_converter])
Action Input: the exact argument to pass to the tool
Observation: the result returned by the tool
... (this Thought/Action/Action Input/Observation can repeat N times until you have enough information)
Thought: I now know the final answer
Final Answer: the complete final response to the user's question

Begin!
"""
print("ReAct System Prompt defined successfully.")


ReAct System Prompt defined successfully.


## 4. ReAct Agent Loop Implementation


In [4]:
def parse_react_output(text: str):
    """Parses model output to extract Thought, Action, Action Input, or Final Answer."""
    if "Final Answer:" in text:
        final_answer = text.split("Final Answer:")[-1].strip()
        return {"type": "final", "answer": final_answer}
    
    action_match = re.search(r"Action:\s*([a-zA-Z0-9_]+)", text)
    input_match = re.search(r"Action Input:\s*(.+)", text)
    thought_match = re.search(r"Thought:\s*(.+)", text)
    
    thought = thought_match.group(1).strip() if thought_match else ""
    
    if action_match and input_match:
        action = action_match.group(1).strip()
        action_input = input_match.group(1).strip().strip("'\"")
        return {
            "type": "action",
            "thought": thought,
            "action": action,
            "action_input": action_input
        }
    
    return {"type": "unknown", "text": text}

def run_react_agent(user_question: str, max_steps: int = 5, verbose: bool = True):
    """Executes the ReAct loop for a given user question and records reasoning traces."""
    trace_log = []
    
    if verbose:
        print(f"\n=======================================================")
        print(f"QUESTION: {user_question}")
        print(f"=======================================================\n")
    
    step = 0
    prompt_accumulator = f"Question: {user_question}\n"
    
    while step < max_steps:
        step += 1
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": REACT_SYSTEM_PROMPT},
                {"role": "user", "content": prompt_accumulator}
            ],
            temperature=0,
            stop=["Observation:"]
        )
        
        output_text = response.choices[0].message.content.strip()
        
        if verbose:
            print(f"--- [Step {step}] ---")
            print(output_text)
            
        parsed = parse_react_output(output_text)
        
        if parsed["type"] == "final":
            trace_log.append({"step": step, "type": "final_answer", "output": parsed["answer"]})
            if verbose:
                print(f"\n FINAL ANSWER:\n{parsed['answer']}\n")
            return parsed["answer"], trace_log
        
        elif parsed["type"] == "action":
            tool_name = parsed["action"]
            tool_input = parsed["action_input"]
            
            trace_entry = {
                "step": step,
                "thought": parsed["thought"],
                "action": tool_name,
                "action_input": tool_input
            }
            
            if tool_name in AVAILABLE_TOOLS:
                tool_func = AVAILABLE_TOOLS[tool_name]
                observation = tool_func(tool_input)
            else:
                observation = f"Error: Tool '{tool_name}' does not exist. Choose from {list(AVAILABLE_TOOLS.keys())}."
            
            if verbose:
                print(f"Observation: {observation}\n")
                
            trace_entry["observation"] = observation
            trace_log.append(trace_entry)
            
            prompt_accumulator += f"{output_text}\nObservation: {observation}\n"
            
        else:
            if verbose:
                print(f"Could not parse action. Output was:\n{output_text}\n")
            break
            
    return "Agent reached max steps without final answer.", trace_log

print("ReAct Runner Function ready!")


ReAct Runner Function ready!


## 5. Multi-Step Test Scenarios & Reasoning Traces


In [5]:
# Test Scenario 1: Financial Comparison & Math Reasoning
q1 = "What is the difference between Apple's Q4 revenue and Microsoft's Q4 revenue in billions of USD?"
ans1, trace1 = run_react_agent(q1)



QUESTION: What is the difference between Apple's Q4 revenue and Microsoft's Q4 revenue in billions of USD?

--- [Step 1] ---
Thought: I need to find the Q4 revenue for both Apple and Microsoft to calculate the difference. I will first search for Apple's Q4 revenue and then for Microsoft's Q4 revenue. 

Action: search_knowledge_base  
Action Input: "Apple Q4 revenue 2023"
Observation: Apple Inc. reported a Q4 revenue of $89.5 billion and net income of $23.0 billion.

--- [Step 2] ---
Thought: I have found Apple's Q4 revenue, which is $89.5 billion. Now, I need to find Microsoft's Q4 revenue for 2023.

Action: search_knowledge_base  
Action Input: "Microsoft Q4 revenue 2023"
Observation: Microsoft Corporation reported a Q4 revenue of $56.5 billion and net income of $22.3 billion.

--- [Step 3] ---
Thought: I now have both companies' Q4 revenues: Apple's is $89.5 billion and Microsoft's is $56.5 billion. I need to calculate the difference between these two amounts.

Action: calculator  


In [6]:
# Test Scenario 2: Multi-Hop Geography & Unit Conversion
q2 = "What is the distance between Paris and London in miles?"
ans2, trace2 = run_react_agent(q2)



QUESTION: What is the distance between Paris and London in miles?

--- [Step 1] ---
Thought: To find the distance between Paris and London, I will search the knowledge base for the specific distance between these two cities. 
Action: search_knowledge_base
Action Input: "distance between Paris and London in miles"
Observation: Paris is the capital of France. The distance between Paris and London is 344 kilometers.

--- [Step 2] ---
Thought: I have found that the distance between Paris and London is 344 kilometers. Now, I need to convert this distance from kilometers to miles.
Action: unit_converter
Action Input: "344 km to miles"
Observation: 344.0 km = 213.75 miles

--- [Step 3] ---
Thought: I now know the final answer.
Final Answer: The distance between Paris and London is approximately 213.75 miles.

 FINAL ANSWER:
The distance between Paris and London is approximately 213.75 miles.



In [7]:
# Test Scenario 3: Complex 3-Step Query (Search -> Calculate -> Convert)
q3 = "Find the total combined Q4 revenue of Tesla and Microsoft in billions of USD, and convert that total to Indian Rupees (INR)."
ans3, trace3 = run_react_agent(q3)



QUESTION: Find the total combined Q4 revenue of Tesla and Microsoft in billions of USD, and convert that total to Indian Rupees (INR).

--- [Step 1] ---
Thought: I need to find the Q4 revenue for both Tesla and Microsoft in billions of USD. After obtaining these figures, I will sum them up to get the total combined revenue. Finally, I will convert this total from USD to Indian Rupees (INR).

Action: search_knowledge_base
Action Input: "Tesla Q4 revenue 2023"
Observation: Tesla Inc. reported a Q4 revenue of $25.17 billion and net income of $7.9 billion.

--- [Step 2] ---
Thought: I have found the Q4 revenue for Tesla, which is $25.17 billion. Now, I need to find the Q4 revenue for Microsoft in 2023.

Action: search_knowledge_base
Action Input: "Microsoft Q4 revenue 2023"
Observation: Microsoft Corporation reported a Q4 revenue of $56.5 billion and net income of $22.3 billion.

--- [Step 3] ---
Thought: I have now obtained the Q4 revenue for both Tesla ($25.17 billion) and Microsoft ($5

## 6. Summary of Reasoning Traces


In [8]:
print("=======================================================")
print(" REASONING TRACE SUMMARY LOG")
print("=======================================================")

scenarios = [
    ("Scenario 1 (Math Difference)", q1, trace1),
    ("Scenario 2 (Unit Conversion)", q2, trace2),
    ("Scenario 3 (Search -> Calculate -> Convert)", q3, trace3)
]

for name, q, trace in scenarios:
    print(f"\n{name}: '{q}'")
    print(f"Total Steps: {len(trace)}")
    for t in trace:
        if t.get("type") == "final_answer":
            print(f"  └─ Step {t['step']} [Final Answer]: {t['output'][:80]}...")
        else:
            print(f"  └─ Step {t['step']} [Action]: {t['action']}({t['action_input']}) -> Obs: {t['observation']}")


 REASONING TRACE SUMMARY LOG

Scenario 1 (Math Difference): 'What is the difference between Apple's Q4 revenue and Microsoft's Q4 revenue in billions of USD?'
Total Steps: 4
  └─ Step 1 [Action]: search_knowledge_base(Apple Q4 revenue 2023) -> Obs: Apple Inc. reported a Q4 revenue of $89.5 billion and net income of $23.0 billion.
  └─ Step 2 [Action]: search_knowledge_base(Microsoft Q4 revenue 2023) -> Obs: Microsoft Corporation reported a Q4 revenue of $56.5 billion and net income of $22.3 billion.
  └─ Step 3 [Action]: calculator(89.5 - 56.5) -> Obs: 33.0
  └─ Step 4 [Final Answer]: The difference between Apple's Q4 revenue and Microsoft's Q4 revenue in 2023 is ...

Scenario 2 (Unit Conversion): 'What is the distance between Paris and London in miles?'
Total Steps: 3
  └─ Step 1 [Action]: search_knowledge_base(distance between Paris and London in miles) -> Obs: Paris is the capital of France. The distance between Paris and London is 344 kilometers.
  └─ Step 2 [Action]: unit_converte

## 7. Conclusion

In this task, a **ReAct (Reasoning + Acting)** Agent was constructed from scratch:

1. **ReAct Paradigm**: The LLM dynamically interleaves step-by-step reasoning (`Thought`) with tool executions (`Action` & `Action Input`), followed by examining results (`Observation`).
2. **Multi-Hop Execution**:
   - Multi-step financial comparison (Knowledge Base Search $\rightarrow$ Calculator).
   - Distance conversion (Knowledge Base Search $\rightarrow$ Unit Converter).
   - 3-step pipeline (Knowledge Base Search for 2 companies $\rightarrow$ Addition via Calculator $\rightarrow$ Currency Conversion via Unit Converter).
3. **Trace Logging**: Complete transparency into the reasoning steps and intermediate tool states.
